In [7]:
import pandas as pd
import numpy as np

# Load datasets
cards = pd.read_csv("Bank_data.csv")
customers = pd.read_csv("customer_expense_dataset_1500.csv")

# Preview
print(cards.head())
#print(customers.head())

  Bank_Name                  Card_Name  Annual_Fee  \
0     ICICI  platinum ship credit card           0   
1     ICICI      Amazon pay icici card           0   
2     ICICI          Coral Credit Card         500   
3      IDFC                  Millennia           0   
4      IDFC              First Classic           0   

   Grocery_Cashback(%) [Zepto, BigBasket, etc.]  Travel_Cashback(%)  \
0                                           1.5                 2.0   
1                                           3.0                 5.0   
2                                           5.0                10.0   
3                                           1.0                 0.0   
4                                           0.0                 0.0   

   Dining_Cashback [Swiggy, Zomato, etc.](%) Min_Income_Required(Yearly)  \
0                                       15.0                      300000   
1                                        3.0                      300000   
2                   

In [8]:
# Rename columns for easier access
cards = cards.rename(columns={
    'Grocery_Cashback(%) [Zepto, BigBasket, etc.]':'grocery_cb',
    'Travel_Cashback(%)':'travel_cb',
    'Dining_Cashback [Swiggy, Zomato, etc.](%)':'dining_cb',
    'Min_Income_Required(Yearly)':'min_income',
    'Limit(montly)':'credit_limit'
})

# Fill missing values
cards.fillna(0, inplace=True)
customers.fillna(0, inplace=True)

# Check data
print(cards.info())
print(customers.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 78 entries, 0 to 77
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Bank_Name                  78 non-null     object 
 1   Card_Name                  78 non-null     object 
 2   Annual_Fee                 78 non-null     int64  
 3   grocery_cb                 78 non-null     float64
 4   travel_cb                  78 non-null     float64
 5   dining_cb                  78 non-null     float64
 6   min_income                 78 non-null     object 
 7   credit_limit               78 non-null     object 
 8   Grocery Discount           78 non-null     int64  
 9   Travel_Discount            78 non-null     int64  
 10  Dining_Discount            78 non-null     int64  
 11  Rewards Points             78 non-null     float64
 12  Review InPoints(out of 5)  78 non-null     float64
 13  CardType                   78 non-null     object 
 

In [9]:
#Best Credit Card for Grocery Shoppers

best_grocery_card = cards.sort_values(by="grocery_cb", ascending=False)

print(best_grocery_card[['Bank_Name','Card_Name','grocery_cb']].head())

         Bank_Name                    Card_Name  grocery_cb
27  Bank of Baroda            IRCTC Credit Card        10.0
28  Bank of Baroda            Prime Credit Card        10.0
26  Bank of Baroda             HPCL Credit Card        10.0
44       Axis Bank  Axis Bank Value+ Debit Card         5.0
2            ICICI            Coral Credit Card         5.0


In [10]:
#Cards with No Annual Fee

no_fee_cards = cards[cards['Annual_Fee'] == 0]

print(no_fee_cards[['Bank_Name','Card_Name','Annual_Fee']])

              Bank_Name                              Card_Name  Annual_Fee
0                 ICICI              platinum ship credit card           0
1                 ICICI                  Amazon pay icici card           0
3                  IDFC                              Millennia           0
4                  IDFC                          First Classic           0
6                  IDFC                           First Select           0
8                  IDFC                           Republic Day           0
11                  PNB                         Global Classic           0
13                  PNB                        Patanjali RuPay           0
28       Bank of Baroda                      Prime Credit Card           0
29          Canara Bank               VISA Classic Credit Card           0
30          Canara Bank                    MasterCard Standard           0
31          Canara Bank             RuPay Platinum Credit Card           0
33          Canara Bank  

In [11]:
#Match Cards Based on Customer Income


cards['min_income'] = pd.to_numeric(cards['min_income'], errors='coerce')

def eligible_cards(customer_income):
    eligible = cards[cards['min_income'] <= customer_income]
    return eligible[['Bank_Name','Card_Name','min_income']]

print(eligible_cards(400000))

              Bank_Name                              Card_Name  min_income
0                 ICICI              platinum ship credit card    300000.0
1                 ICICI                  Amazon pay icici card    300000.0
3                  IDFC                              Millennia    300000.0
7                  IDFC                                    Ace    300000.0
8                  IDFC                           Republic Day    200000.0
10                  PNB                            Global Gold    350000.0
11                  PNB                         Global Classic    250000.0
12                  PNB                             Millennial    300000.0
13                  PNB                        Patanjali RuPay    100000.0
14                  PNB                           RuPay Select         0.0
15                IDBI                IDBI Delight Credit Card    240000.0
18                  SBI                      Cashback SBI Card    300000.0
19                  SBI  

In [14]:
#Calculate Total Cashback for a Customer

def calculate_cashback(customer):

    grocery = customer['Grocery']
    travel = customer['Travelling']
    dining = customer['Dining']

    cashback = cards.copy()

    cashback['Grocery_CB_Value'] = grocery * cashback['grocery_cb']/100
    cashback['Travel_CB_Value'] = travel * cashback['travel_cb']/100
    cashback['Dining_CB_Value'] = dining * cashback['dining_cb']/100

    cashback['Total_Cashback'] = (
        cashback['Grocery_CB_Value'] +
        cashback['Travel_CB_Value'] +
        cashback['Dining_CB_Value']
    )

    return cashback[['Bank_Name','Card_Name','Total_Cashback']].sort_values(
        by='Total_Cashback', ascending=False
    )


customer = customers.iloc[0]
print(calculate_cashback(customer).head())

         Bank_Name                 Card_Name  Total_Cashback
27  Bank of Baroda         IRCTC Credit Card          1094.5
28  Bank of Baroda         Prime Credit Card          1094.5
26  Bank of Baroda          HPCL Credit Card          1094.5
2            ICICI         Coral Credit Card           775.2
15           IDBI   IDBI Delight Credit Card           751.9


In [15]:
#Best Dining Cashback Cards

best_dining = cards.sort_values(by="dining_cb", ascending=False)

print(best_dining[['Bank_Name','Card_Name','dining_cb']].head())

    Bank_Name                     Card_Name  dining_cb
43  Axis Bank  Axis Bank Delight Debit Card       25.0
2       ICICI             Coral Credit Card       20.0
0       ICICI     platinum ship credit card       15.0
15      IDBI       IDBI Delight Credit Card       15.0
16      IDBI          Royale Signature card       15.0


In [16]:
#Annual Fee vs Cashback Comparison

cashback = calculate_cashback(customers.iloc[0])

comparison = cashback.merge(cards[['Card_Name','Annual_Fee']], on="Card_Name")

comparison['Net_Value'] = comparison['Total_Cashback'] - comparison['Annual_Fee']

print(comparison.sort_values(by="Net_Value", ascending=False).head())

         Bank_Name              Card_Name  Total_Cashback  Annual_Fee  \
1   Bank of Baroda      Prime Credit Card         1094.50         0.0   
2   Bank of Baroda       HPCL Credit Card         1094.50       499.0   
0   Bank of Baroda      IRCTC Credit Card         1094.50       500.0   
12             PNB         Global Classic          397.88         0.0   
13           ICICI  Amazon pay icici card          369.28         0.0   

    Net_Value  
1     1094.50  
2      595.50  
0      594.50  
12     397.88  
13     369.28  


In [18]:
#Filter Cards by Credit Limit

cards['credit_limit'] = pd.to_numeric(cards['credit_limit'], errors='coerce')

cards[cards['credit_limit'] >= 50000][['Bank_Name','Card_Name','credit_limit']]

,Bank_Name,Card_Name,credit_limit
0,ICICI,platinum ship credit card,60000.0
1,ICICI,Amazon pay icici card,100000.0
2,ICICI,Coral Credit Card,80000.0
3,IDFC,Millennia,125000.0
4,IDFC,First Classic,150000.0
...,...,...,...
73,Kotak Mahindra Bank,IndianOil Kotak Credit Card,300000.0
74,Kotak Mahindra Bank,Zen Signature Credit Card,600000.0
75,Kotak Mahindra Bank,Kotak Air Credit Card,500000.0
76,Kotak Mahindra Bank,PVR INOX Kotak Credit Card,200000.0


In [19]:
#Highest Reward Points Cards

cards.sort_values(by="Rewards Points", ascending=False)[
    ['Bank_Name','Card_Name','Rewards Points']
].head()

,Bank_Name,Card_Name,Rewards Points
38,Axis Bank,Axis Bank Neo Credit Card,200.0
13,PNB,Patanjali RuPay,150.0
3,IDFC,Millennia,150.0
9,PNB,Global Platinum,150.0
10,PNB,Global Gold,100.0


In [20]:
#Cards with Lounge Access

lounge_cards = cards[cards['Lounge access per year'] > 0]

print(lounge_cards[['Bank_Name','Card_Name','Lounge access per year']])

              Bank_Name                              Card_Name  \
2                 ICICI                      Coral Credit Card   
3                  IDFC                              Millennia   
4                  IDFC                          First Classic   
5                  IDFC                            WOW ! Black   
6                  IDFC                           First Select   
8                  IDFC                           Republic Day   
9                   PNB                        Global Platinum   
10                  PNB                            Global Gold   
12                  PNB                             Millennial   
16                IDBI                   Royale Signature card   
18                  SBI                      Cashback SBI Card   
21                  SBI                         SBI Card Prime   
22                  SBI                         SBI Card ELITE   
25                  SBI  SBI Platinum International Debit Card   
26       B

In [21]:
#Customer Specific Credit Card Recommendation

def recommend_card(customer):

    result = calculate_cashback(customer)
    eligible = cards[cards['min_income'] <= customer['total Income']]
    result = result[result['Card_Name'].isin(eligible['Card_Name'])]
    return result.head(5)

print(recommend_card(customers.iloc[10]))

         Bank_Name                     Card_Name  Total_Cashback
28  Bank of Baroda             Prime Credit Card         1847.10
48       HDFC Bank                        Swiggy         1159.84
13             PNB               Patanjali RuPay          826.31
44       Axis Bank   Axis Bank Value+ Debit Card          739.50
43       Axis Bank  Axis Bank Delight Debit Card          682.75
